# T-Rex PPO Training (Colab)

Train a T-Rex to walk, run, and bite prey using PPO with curriculum learning.

**Training Stages:**
1. **Balance** - Stand without falling
2. **Locomotion** - Walk and run forward
3. **Bite** - Sprint toward prey and bite with jaw

Uses the custom `TRexEnv` from `mesozoic-labs` with MuJoCo + Gymnasium + Stable-Baselines3.

In [ ]:
# Install dependencies and set up GPU rendering
!pip install mujoco

from google.colab import files
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
    raise RuntimeError(
        'Cannot communicate with GPU. '
        'Make sure you are using a GPU Colab runtime. '
        'Go to the Runtime menu and select Choose runtime type.')

# Add EGL ICD config for Nvidia GPU rendering
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

%env MUJOCO_GL=egl

# Verify MuJoCo
import mujoco
mujoco.MjModel.from_xml_string('<mujoco/>')
print(f'MuJoCo {mujoco.__version__} installed successfully.')

# Install media tools
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy

import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
clear_output()
print('Setup complete.')

In [ ]:
# Clone mesozoic-labs and install
!pip install stable-baselines3[extra]
!git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo "Already cloned"
!pip install -e /content/mesozoic-labs

from IPython.display import clear_output
clear_output()
print('mesozoic-labs installed.')

In [ ]:
import gymnasium
import mujoco
import numpy as np
import os
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecVideoRecorder
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed

from environments.trex.envs.trex_env import TRexEnv

print(f"MuJoCo: {mujoco.__version__}")
print(f"Gymnasium: {gymnasium.__version__}")
print(f"TRexEnv loaded successfully")

## Explore the T-Rex Environment

In [ ]:
env = TRexEnv()

print(f"Observation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")

model = env.model
print(f"\nModel info:")
print(f"  Bodies: {model.nbody}")
print(f"  Joints: {model.njnt}")
print(f"  Actuators: {model.nu}")
print(f"  Total mass: {sum(model.body_mass):.2f} kg")

print(f"\nActuators:")
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  [{i:2d}] {name}")

env.close()

## Curriculum Stage Configurations

In [ ]:
STAGE_CONFIGS = {
    1: {
        "name": "balance",
        "description": "Learn to stand without falling",
        "env_kwargs": {
            "forward_vel_weight": 0.0,
            "alive_bonus": 1.0,
            "energy_penalty_weight": 0.0005,
            "tail_stability_weight": 0.1,
            "bite_bonus": 0.0,
            "bite_approach_weight": 0.0,
            "prey_distance_range": (10.0, 15.0),
            "max_episode_steps": 500,
        },
        "ppo_kwargs": {
            "learning_rate": 3e-4,
            "n_steps": 2048,
            "batch_size": 64,
            "n_epochs": 10,
            "gamma": 0.99,
            "gae_lambda": 0.95,
            "clip_range": 0.2,
            "ent_coef": 0.01,
        },
        "timesteps": 500_000,
    },
    2: {
        "name": "locomotion",
        "description": "Walk and run forward",
        "env_kwargs": {
            "forward_vel_weight": 1.0,
            "alive_bonus": 0.5,
            "energy_penalty_weight": 0.001,
            "tail_stability_weight": 0.05,
            "bite_bonus": 0.0,
            "bite_approach_weight": 0.2,
            "prey_distance_range": (8.0, 12.0),
            "max_episode_steps": 1000,
        },
        "ppo_kwargs": {
            "learning_rate": 1e-4,
            "n_steps": 2048,
            "batch_size": 128,
            "n_epochs": 10,
            "gamma": 0.99,
            "gae_lambda": 0.95,
            "clip_range": 0.2,
            "ent_coef": 0.005,
        },
        "timesteps": 1_000_000,
    },
    3: {
        "name": "bite",
        "description": "Sprint and bite prey with jaw",
        "env_kwargs": {
            "forward_vel_weight": 1.0,
            "alive_bonus": 0.1,
            "energy_penalty_weight": 0.001,
            "tail_stability_weight": 0.02,
            "bite_bonus": 500.0,
            "bite_approach_weight": 0.5,
            "prey_distance_range": (3.0, 8.0),
            "prey_lateral_range": (-1.5, 1.5),
            "max_episode_steps": 1000,
        },
        "ppo_kwargs": {
            "learning_rate": 5e-5,
            "n_steps": 4096,
            "batch_size": 256,
            "n_epochs": 10,
            "gamma": 0.995,
            "gae_lambda": 0.95,
            "clip_range": 0.1,
            "ent_coef": 0.001,
        },
        "timesteps": 2_000_000,
    },
}

for stage, config in STAGE_CONFIGS.items():
    print(f"Stage {stage}: {config['name']} - {config['description']}")
    print(f"  Timesteps: {config['timesteps']:,}")
    print()

## Training Helpers

In [ ]:
N_ENVS = 4
SEED = 42
log_dir = "./logs/trex_ppo"

def make_env(stage, rank, seed=0):
    """Create a single TRexEnv instance."""
    def _init():
        env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
        env = TRexEnv(**env_kwargs)
        env = Monitor(env)
        env.reset(seed=seed + rank)
        return env
    set_random_seed(seed)
    return _init

def create_vec_env(stage, n_envs=N_ENVS, seed=SEED):
    """Create vectorized environment with observation/reward normalization."""
    env = DummyVecEnv([make_env(stage, i, seed) for i in range(n_envs)])
    env = VecNormalize(env, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, clip_reward=10.0)
    return env

print("Training utilities ready.")

## Train

Set `STAGE` to 1, 2, or 3. For quick testing set `QUICK_TEST = True`.

In [ ]:
STAGE = 1
QUICK_TEST = True
TIMESTEPS = 50_000 if QUICK_TEST else STAGE_CONFIGS[STAGE]["timesteps"]

print(f"Training Stage {STAGE}: {STAGE_CONFIGS[STAGE]['name']}")
print(f"Timesteps: {TIMESTEPS:,}")

# Create environments
train_env = create_vec_env(STAGE)
eval_env = create_vec_env(STAGE, n_envs=1, seed=SEED + 1000)

# Create PPO model
config = STAGE_CONFIGS[STAGE]
ppo_kwargs = config["ppo_kwargs"].copy()
ppo_kwargs["verbose"] = 1

model = PPO("MlpPolicy", train_env, **ppo_kwargs)

# Callbacks
stage_dir = os.path.join(log_dir, f"stage{STAGE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(stage_dir, exist_ok=True)
model_dir = os.path.join(stage_dir, "models")
os.makedirs(model_dir, exist_ok=True)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=model_dir,
    log_path=stage_dir,
    eval_freq=max(5000 // N_ENVS, 1),
    n_eval_episodes=3,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(10000 // N_ENVS, 1),
    save_path=model_dir,
    name_prefix=f"stage{STAGE}",
)

# Train
print(f"\nTraining for {TIMESTEPS:,} timesteps...")
model.learn(
    total_timesteps=TIMESTEPS,
    callback=CallbackList([eval_callback, checkpoint_callback]),
    progress_bar=True,
)

# Save
final_path = os.path.join(model_dir, f"stage{STAGE}_final")
model.save(final_path)
train_env.save(final_path + "_vecnorm.pkl")
print(f"\nModel saved to: {final_path}.zip")

# Evaluate
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=5)
print(f"Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

train_env.close()
eval_env.close()

## Training Curve

In [ ]:
eval_log = os.path.join(stage_dir, "evaluations.npz")

if os.path.exists(eval_log):
    data = np.load(eval_log)
    timesteps = data["timesteps"]
    results = data["results"]
    mean_rewards = np.mean(results, axis=1)
    std_rewards = np.std(results, axis=1)

    plt.figure(figsize=(10, 5))
    plt.plot(timesteps, mean_rewards, 'b-', label='Mean Reward')
    plt.fill_between(timesteps,
                     mean_rewards - std_rewards,
                     mean_rewards + std_rewards,
                     alpha=0.3)
    plt.xlabel('Timesteps')
    plt.ylabel('Reward')
    plt.title(f'T-Rex PPO - Stage {STAGE} Training Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No evaluation log found.")